# **Contruction of the 5 languages**

## 1. Lexic and English

In [ ]:
from __future__ import annotations
from dataclasses import dataclass
from typing import List, Dict, Optional
import os
import json
import shutil
import pandas as pd


# Language A = isolating: free grammatical particles, no affixes
# Language B = analytic: auxiliary-like words and classifier-like plural marking
# Language C = fusional prefixing: one prefix fuses person and tense
# Language D = agglutinative suffixing: transparent suffix chains
# Language E = polysynthetic mixed: subject prefix + incorporated nouns + tense suffix


LEX = {
    # pronouns / participants
    "I": "ma",
    "you": "ta",
    "he": "lo",
    "she": "lo",
    "it": "lo",

    # verbs
    "go": "daka",
    "cry": "naku",
    "eat": "notu",
    "see": "luma",
    "rain": "bogu",
    "be": "ge",

    # nouns
    "city": "kano",
    "house": "toto",
    "person": "mira",
    "tree": "saki",
    "food": "peli",
    "bread": "fenu",
    "fruit": "raki",
    "meal": "dovu",

    # adjectives
    "big": "kiwa",
    "small": "pika",
    "sad": "sima",

    # adverbs / function words
    "here": "nala",
    "tomorrow": "fila",
    "yesterday": "sena",
    "because": "ka",
    "very": "po",
}

def root(x: Optional[str]) -> Optional[str]:
    if x is None:
        return None
    return LEX[x]


def join_nonempty(parts: List[Optional[str]], sep: str = " ") -> str:
    return sep.join([p for p in parts if p])


def cap(s: str) -> str:
    return s[0].upper() + s[1:] if s else s

@dataclass
class Meaning:
    english: str
    clause_type: str                     # intransitive / transitive / copular / weather / causal
    subject: Optional[str] = None
    verb: Optional[str] = None
    obj: Optional[str] = None
    plural_object: bool = False
    adjective: Optional[str] = None
    tense: str = "present"               # present / past / future
    adverb: Optional[str] = None          # here / tomorrow / yesterday / very
    location: Optional[str] = None
    location_role: Optional[str] = None   # to / into / in
    cause: Optional["Meaning"] = None


# english generation functions
def english_subject_form(subj: str) -> str:
    return {"I": "I", "you": "You", "he": "He", "she": "She", "it": "It"}[subj]

# The verb "be" is highly irregular in English, so we need a special function to handle it.
def english_be(subj: str, tense: str = "present") -> str:
    if tense == "present":
        return {"I": "am", "you": "are", "he": "is", "she": "is", "it": "is"}[subj]
    if tense == "past":
        return {"I": "was", "you": "were", "he": "was", "she": "was", "it": "was"}[subj]
    return "will be"


# For regular verbs, we can use a simple function to get the present and past forms. For simplicity, we'll only handle a few verbs and ignore irregularities beyond "be".
def english_present_verb(verb: str, subj: str) -> str:
    if subj in {"he", "she", "it"}:
        return {
            "go": "goes",
            "cry": "cries",
            "eat": "eats",
            "see": "sees",
            "rain": "rains",
        }[verb]
    return verb


# For the past tense, we'll just hardcode the forms for our verbs.
def english_past_verb(verb: str) -> str:
    return {
        "go": "went",
        "cry": "cried",
        "eat": "ate",
        "see": "saw",
        "rain": "rained",
    }[verb]

# English nouns also have some irregular plural forms, so we'll handle those as well.
def english_noun_phrase(noun: str, plural: bool = False) -> str:
    forms = {
        "city": ("the city", "the cities"),
        "house": ("the house", "the houses"),
        "person": ("the person", "the people"),
        "tree": ("the tree", "the trees"),
        "meal": ("the meal", "the meals"),
        "food": ("food", "food"),
        "bread": ("bread", "bread"),
        "fruit": ("fruit", "fruit"),
    }
    singular, plural_form = forms[noun]
    return plural_form if plural else singular

# English expresses location with prepositions, so we need to convert our location and location role into a prepositional phrase.
def english_location_phrase(location: Optional[str], role: Optional[str]) -> str:
    if location is None or role is None:
        return ""
    np = english_noun_phrase(location)
    return {"to": f"to {np}", "into": f"into {np}", "in": f"in {np}"}[role]

# For causal clauses, we need to lowercase the start of the clause unless it starts with "I".
def lowercase_clause_start(s: str) -> str:
    if not s:
        return s
    if s.startswith("I "):
        return s
    return s[0].lower() + s[1:]

# Finally, we can put everything together to build the English sentence from the Meaning object.
def build_english(m: Meaning) -> str:
    if m.clause_type == "weather":
        subj = english_subject_form(m.subject)
        if m.tense == "present":
            base = f"{subj} {english_present_verb(m.verb, m.subject)}"
        elif m.tense == "future":
            base = f"{subj} will {m.verb}"
        else:
            base = f"{subj} {english_past_verb(m.verb)}"
        if m.adverb == "tomorrow":
            base += " tomorrow"
        elif m.adverb == "yesterday":
            base += " yesterday"
        return base + "."

# The copular clause is a bit tricky because it can have either a subject or an object, and the verb "be" is irregular. We also need to handle adjectives and the "very" adverb.
    if m.clause_type == "copular":
        if m.subject:
            if m.tense == "future":
                base = f"{english_subject_form(m.subject)} will be"
            else:
                base = f"{english_subject_form(m.subject)} {english_be(m.subject, m.tense)}"
        else:
            np = english_noun_phrase(m.obj, m.plural_object)
            base = f"{cap(np)} {'are' if m.plural_object else 'is'}"
        base += f" {'very ' if m.adverb == 'very' else ''}{m.adjective}"
        return base + "."


# For causal clauses, we build the main clause and the cause clause separately, lowercase the start of the cause clause if necessary, and then join them with "because".
    if m.clause_type == "causal":
        main = build_english(Meaning("", "intransitive", subject=m.subject, verb=m.verb, tense=m.tense)).rstrip(".")
        sub = lowercase_clause_start(build_english(m.cause).rstrip("."))
        return f"{main} because {sub}."

# For transitive and intransitive clauses, we build the sentence based on the subject
    if m.clause_type == "intransitive":
        subj = english_subject_form(m.subject)
        if m.tense == "present":
            base = f"{subj} {english_present_verb(m.verb, m.subject)}"
        elif m.tense == "future":
            base = f"{subj} will {m.verb}"
        else:
            base = f"{subj} {english_past_verb(m.verb)}"
        loc = english_location_phrase(m.location, m.location_role)
        if loc:
            base += f" {loc}"
        if m.adverb == "here":
            base += " here"
        elif m.adverb == "tomorrow":
            base += " tomorrow"
        elif m.adverb == "yesterday":
            base += " yesterday"
        return base + "."

# For transitive clauses, we also need to include the object noun phrase.
    if m.clause_type == "transitive":
        subj = english_subject_form(m.subject)
        if m.tense == "present":
            base = f"{subj} {english_present_verb(m.verb, m.subject)}"
        elif m.tense == "future":
            base = f"{subj} will {m.verb}"
        else:
            base = f"{subj} {english_past_verb(m.verb)}"
        base += f" {english_noun_phrase(m.obj, m.plural_object)}"
        loc = english_location_phrase(m.location, m.location_role)
        if loc:
            base += f" {loc}"
        if m.adverb == "tomorrow":
            base += " tomorrow"
        elif m.adverb == "yesterday":
            base += " yesterday"
        return base + "."

    raise ValueError(f"Unknown clause type: {m.clause_type}")

id	english	language_A	language_B	language_C	language_D	language_E
1	I go.	ma daka	ma daka	medaka	daka-ka	madaka
2	I go to the city.	ma daka du kano	ma daka mu kano	medaka dra kano	daka-ka kano-mu	makanomodaka
3	I go into the house.	ma daka ri toto	ma daka nel toto	medaka vren toto	daka-ka toto-nun	matotonudaka
4	I will go tomorrow.	fila ma vek daka	ma zo daka fila	ardaka fila	daka-ka-ra fila	madakarufila
5	I will go to the city tomorrow.	fila ma vek daka du kano	ma zo daka mu kano fila	ardaka dra kano fila	daka-ka-ra kano-mu fila	makanomodakarufila
6	I will go into the house tomorrow.	fila ma vek daka ri toto	ma zo daka nel toto fila	ardaka vren toto fila	daka-ka-ra toto-nun fila	matotonudakarufila
7	I went yesterday.	sena ma tun daka	ma ki daka sena	ukdaka sena	daka-ka-si sena	madakasesena
8	I went to the city yesterday.	sena ma tun daka du kano	ma ki daka mu kano sena	ukdaka dra kano sena	daka-ka-si kano-mu sena	makanomodakasesena
9	I went into the house yesterday.	sena ma tun daka r

## 2. The languages

### 2.1 LANGUAGE A

In [ ]:
# isolating generation functions
A_TENSE = {"present": None, "future": "vek", "past": "tun"}
A_PLURAL = "nul"
A_LOC = {"to": "du", "into": "ri", "in": "na"}

# The isolating language is the simplest to generate because it just uses free particles and word order, without any affixes. We just need to follow the basic SVO order and add the appropriate particles for tense, plurality, location, and adverbs.
def to_isolating(m: Meaning) -> str:
    parts: List[str] = []

# Adverbs of time like "tomorrow" and "yesterday" can just be added at the end of the sentence.
    if m.adverb == "tomorrow":
        parts.append(root("tomorrow"))
    elif m.adverb == "yesterday":
        parts.append(root("yesterday"))

# Weather clauses have a special word order where the subject comes first, followed by the tense marker (if any), and then the verb. We can just follow this order and add the appropriate particles.
    if m.clause_type == "weather":
        parts.append(root(m.subject))
        if A_TENSE[m.tense]:
            parts.append(A_TENSE[m.tense])
        parts.append(root(m.verb))
        return join_nonempty(parts)

# The copular clause is also a bit special because it can have either a subject or an object, and the verb "be" is just a root with no tense marking. We also need to handle adjectives and the "very" adverb.
    if m.clause_type == "copular":
        if m.subject:
            parts.append(root(m.subject))
        else:
            parts.append(root(m.obj))
            if m.plural_object:
                parts.append(A_PLURAL)
        parts.append(root("be"))
        if m.adverb == "very":
            parts.append(root("very"))
        parts.append(root(m.adjective))
        return join_nonempty(parts)

# For causal clauses, we build the main clause and the cause clause separately and then join them with "because".
    if m.clause_type == "causal":
        main = to_isolating(Meaning("", "intransitive", subject=m.subject, verb=m.verb, tense=m.tense))
        sub = to_isolating(m.cause)
        return f"{main} {root('because')} {sub}"

# For transitive and intransitive clauses, we just follow the SVO order and add particles as needed.
    if m.subject:
        parts.append(root(m.subject))
    if A_TENSE[m.tense]:
        parts.append(A_TENSE[m.tense])
    parts.append(root(m.verb))

# For transitive clauses, we also need to include the object noun phrase.
    if m.obj:
        parts.append(root(m.obj))
        if m.plural_object:
            parts.append(A_PLURAL)

# For location, we add the appropriate particle followed by the location noun.
    if m.location:
        parts.extend([A_LOC[m.location_role], root(m.location)])

# Adverbs of place like "here" can just be added at the end of the sentence.
    if m.adverb == "here":
        parts.append(root("here"))

# Finally, we join all the parts together with spaces, ignoring any None values.
    return join_nonempty(parts)

### 2.2 LANGUAGE B

In [ ]:
# The analytic language is a bit more complex because it has auxiliary-like words for tense and classifier-like plural marking, but it still relies on word order and free particles rather than affixes. We need to handle the tense auxiliaries, plural marking, location particles, and adverbs appropriately while following the basic SVO order.
B_AUX = {"present": None, "future": "zo", "past": "ki"}
B_PLURAL = "enu"
B_LOC = {"to": "mu", "into": "nel", "in": "sha"}

# For nouns, we need to add the plural marker if the noun is plural, but we don't have any affixes, so we just add a separate word for plural.
def b_noun(noun: str, plural: bool = False) -> str:
    return join_nonempty([B_PLURAL if plural else None, root(noun)])

# For the analytic language, we follow a similar structure to the isolating language, but we need to include the tense auxiliaries and plural markers as separate words. The word order is still SVO, but we have more particles to include.
def to_analytic(m: Meaning) -> str:
    parts: List[str] = []

# Weather clauses have a special word order where the subject comes first, followed by the tense auxiliary (if any), and then the verb. We can just follow this order and add the appropriate particles.
    if m.clause_type == "weather":
        parts.append(root(m.subject))
        if B_AUX[m.tense]:
            parts.append(B_AUX[m.tense])
        parts.append(root(m.verb))
        if m.adverb == "tomorrow":
            parts.append(root("tomorrow"))
        elif m.adverb == "yesterday":
            parts.append(root("yesterday"))
        return join_nonempty(parts)

# The copular clause is also a bit special because it can have either a subject or an object, and the verb "be" is just a root with no tense marking. We also need to handle adjectives and the "very" adverb.
    if m.clause_type == "copular":
        head = root(m.subject) if m.subject else b_noun(m.obj, m.plural_object)
        parts.append(head)
        if B_AUX[m.tense]:
            parts.append(B_AUX[m.tense])
        parts.append(root("be"))
        if m.adverb == "very":
            parts.append(root("very"))
        parts.append(root(m.adjective))
        return join_nonempty(parts)

# For causal clauses, we build the main clause and the cause clause separately and then join them with "because".
    if m.clause_type == "causal":
        main = to_analytic(Meaning("", "intransitive", subject=m.subject, verb=m.verb, tense=m.tense))
        sub = to_analytic(m.cause)
        return f"{main} {root('because')} {sub}"

# For transitive and intransitive clauses, we just follow the SVO order and add particles as needed.
    if m.subject:
        parts.append(root(m.subject))
    if B_AUX[m.tense]:
        parts.append(B_AUX[m.tense])
    parts.append(root(m.verb))

# For transitive clauses, we also need to include the object noun phrase.
    if m.obj:
        parts.append(b_noun(m.obj, m.plural_object))

# For location, we add the appropriate particle followed by the location noun.
    if m.location:
        parts.extend([B_LOC[m.location_role], root(m.location)])

# Adverbs of place like "here" can just be added at the end of the sentence.
    if m.adverb == "here":
        parts.append(root("here"))
    elif m.adverb == "tomorrow" and m.tense == "future":
        parts.append(root("tomorrow"))
    elif m.adverb == "yesterday" and m.tense == "past":
        parts.append(root("yesterday"))

# Finally, we join all the parts together with spaces, ignoring any None values.
    return join_nonempty(parts)

### 2.3 LANGUAGE C

In [ ]:
# The fusional prefixing language is more complex because it has prefixes that fuse person and tense together, as well as some irregularities in the copula and plural marking. We need to handle the verb prefixes, copula forms, plural classes, location particles, and adverbs while following the basic SVO order. The verb prefixes are highly fusional, so we have a different prefix for each combination of tense and
C_PREFIXES = {
    ("present", "I"): "me",
    ("present", "you"): "ti",
    ("present", "he"): "na",
    ("present", "she"): "na",
    ("present", "it"): "na",

    ("past", "I"): "uk",
    ("past", "you"): "ot",
    ("past", "he"): "ev",
    ("past", "she"): "ev",
    ("past", "it"): "ev",

    ("future", "I"): "ar",
    ("future", "you"): "el",
    ("future", "he"): "or",
    ("future", "she"): "or",
    ("future", "it"): "or",
}

# The copula is also irregular in this language, with different forms for each combination of tense and subject. We need to handle this separately from the regular verb prefixes.
C_COPULA = {
    ("present", "I"): "mege",
    ("present", "you"): "tige",
    ("present", "he"): "nage",
    ("present", "she"): "nage",
    ("present", "it"): "nage",

    ("past", "I"): "ukge",
    ("past", "you"): "otge",
    ("past", "he"): "evge",
    ("past", "she"): "evge",
    ("past", "it"): "evge",

    ("future", "I"): "arge",
    ("future", "you"): "elge",
    ("future", "he"): "orge",
    ("future", "she"): "orge",
    ("future", "it"): "orge",
}

# Plural is also mildly fusional: different noun classes take different plural endings.
C_PLURAL_CLASSES = {
    "city": "esh",
    "house": "um",
    "person": "ai",
    "tree": "ok",
    "meal": "ir",
}

# Location particles are also irregular and don't follow a consistent pattern, so we need to handle them separately as well.
C_LOC = {"to": "dra", "into": "vren", "in": "nim"}

# For the fusional prefixing language, we need to handle the verb prefixes that fuse person and tense together, as well as the irregular copula forms and plural marking. The word order is still SVO, but we have to be careful about how we construct the verb forms and noun phrases.
def c_verb(verb: str, tense: str, subj: str) -> str:
    return C_PREFIXES[(tense, subj)] + root(verb)

# The copula is irregular, so we need a separate function to get the correct form based on tense and subject.
def c_copula(tense: str, subj: str) -> str:
    return C_COPULA[(tense, subj)]

# For nouns, we need to add the plural marker based on the noun class, but since it's a fusional language, we just add the appropriate suffix to the root.
def c_noun(noun: str, plural: bool = False) -> str:
    base = root(noun)
    if not plural:
        return base
    return base + C_PLURAL_CLASSES.get(noun, "ir")

# For the fusional prefixing language, we follow a similar structure to the isolating and analytic languages, but we need to include the verb prefixes that fuse person and tense together, as well as the irregular copula forms and plural marking. The word order is still SVO, but we have to be careful about how we construct the verb forms and noun phrases.
def to_fusional_prefixing(m: Meaning) -> str:
    if m.clause_type == "weather":
        pred = c_verb(m.verb, m.tense, m.subject)
        if m.adverb == "tomorrow":
            return join_nonempty([pred, root("tomorrow")])
        if m.adverb == "yesterday":
            return join_nonempty([pred, root("yesterday")])
        return pred

# The copular clause is also a bit special because it can have either a subject or an object, and the verb "be" is irregular. We also need to handle adjectives and the "very" adverb.  
    if m.clause_type == "copular":
        if m.subject:
            head = root(m.subject)
            cop = c_copula(m.tense, m.subject)
        else:
            head = c_noun(m.obj, m.plural_object)
            cop = c_copula("present", "he")
        adj = root(m.adjective)
        if m.adverb == "very":
            adj = root("very") + adj
        return join_nonempty([head, cop, adj])

# For causal clauses, we build the main clause and the cause clause separately and then join them with "because".
    if m.clause_type == "causal":
        main = to_fusional_prefixing(Meaning("", "intransitive", subject=m.subject, verb=m.verb, tense=m.tense))
        sub = to_fusional_prefixing(m.cause)
        return f"{main} {root('because')} {sub}"

    parts = [c_verb(m.verb, m.tense, m.subject)]

# For transitive clauses, we also need to include the object noun phrase.
    if m.obj:
        parts.append(c_noun(m.obj, m.plural_object))

# For location, we add the appropriate particle followed by the location noun.
    if m.location:
        parts.extend([C_LOC[m.location_role], root(m.location)])

# Adverbs of place like "here" can just be added at the end of the sentence.
    if m.adverb == "here":
        parts.append(root("here"))
    elif m.adverb == "tomorrow" and m.tense == "future":
        parts.append(root("tomorrow"))
    elif m.adverb == "yesterday" and m.tense == "past":
        parts.append(root("yesterday"))

# Finally, we join all the parts together with spaces, ignoring any None values.
    return join_nonempty(parts)

### 2.4 LANGUAGE D

In [ ]:
# The agglutinative suffixing language is the most complex because it has transparent suffix chains for person, tense, plurality, and case, as well as some irregularities in the copula. We need to handle the verb suffixes, copula forms, plural marking, location case markers, and adverbs while following the basic SVO order. The verb suffixes are agglutinative, so we just add the appropriate suffixes to the root verb based on tense and subject.
D_PERSON = {"I": "ka", "you": "tu", "he": "li", "she": "li", "it": "li"}
D_TENSE = {"present": None, "future": "ra", "past": "si"}
D_PLURAL = "min"
D_CASE = {"to": "mu", "into": "nun", "in": "ne"}

# For the agglutinative suffixing language, we need to handle the verb suffixes that indicate person and tense, as well as the irregular copula forms and plural marking. The word order is still SVO, but we have to be careful about how we construct the verb forms and noun phrases with the appropriate suffixes.
def d_verb(verb: str, tense: str, subj: str) -> str:
    parts = [root(verb), D_PERSON[subj]]
    if D_TENSE[tense]:
        parts.append(D_TENSE[tense])
    return "-".join(parts)

# For nouns, we need to add the plural marker as a separate suffix, and for location, we need to add the appropriate case marker as a suffix to the location noun.
def d_noun(noun: str, plural: bool = False, role: Optional[str] = None) -> str:
    parts = [root(noun)]
    if plural:
        parts.append(D_PLURAL)
    if role:
        parts.append(D_CASE[role])
    return "-".join(parts)

# For the agglutinative suffixing language, we follow a similar structure to the other languages, but we need to include the verb suffixes that indicate person and tense, as well as the irregular copula forms and plural marking. The word order is still SVO, but we have to be careful about how we construct the verb forms and noun phrases with the appropriate suffixes.
def to_agglutinative_suffixing(m: Meaning) -> str:
    if m.clause_type == "weather":
        pred = d_verb(m.verb, m.tense, m.subject)
        if m.adverb == "tomorrow":
            return join_nonempty([pred, root("tomorrow")])
        if m.adverb == "yesterday":
            return join_nonempty([pred, root("yesterday")])
        return pred

# The copular clause is also a bit special because it can have either a subject or an object, and the verb "be" is irregular. We also need to handle adjectives and the "very" adverb.
    if m.clause_type == "copular":
        if m.subject:
            head = root(m.subject)
        else:
            head = d_noun(m.obj, m.plural_object)
        adj = root(m.adjective)
        if m.adverb == "very":
            adj = root("very") + "-" + adj
        return join_nonempty([head, adj])

# For causal clauses, we build the main clause and the cause clause separately and then join them with "because".
    if m.clause_type == "causal":
        main = to_agglutinative_suffixing(Meaning("", "intransitive", subject=m.subject, verb=m.verb, tense=m.tense))
        sub = to_agglutinative_suffixing(m.cause)
        return f"{main} {root('because')} {sub}"

    parts = [d_verb(m.verb, m.tense, m.subject)]

# For transitive clauses, we also need to include the object noun phrase.
    if m.obj:
        parts.append(d_noun(m.obj, m.plural_object))

# For location, we add the appropriate case marker as a suffix to the location noun.
    if m.location:
        parts.append(d_noun(m.location, False, m.location_role))

# Adverbs of place like "here" can just be added at the end of the sentence.
    if m.adverb == "here":
        parts.append(root("here"))
    elif m.adverb == "tomorrow" and m.tense == "future":
        parts.append(root("tomorrow"))
    elif m.adverb == "yesterday" and m.tense == "past":
        parts.append(root("yesterday"))

# Finally, we join all the parts together with hyphens, ignoring any None values.
    return join_nonempty(parts)

### 2.5 LANGUAGE E

In [ ]:
# The polysynthetic mixed language is the most complex because it has a combination of subject prefixes, incorporated nouns, tense suffixes, and irregularities in the copula and location incorporation. We need to handle the verb forms with subject prefixes and tense suffixes, noun incorporation for objects and locations, and the irregular copula forms while following the basic SVO order. The verb forms are highly polysynthetic, so we have a subject prefix that indicates person and a tense suffix that indicates tense, with the root verb in between.
E_SUBJECT_PREFIX = {"I": "ma", "you": "ta", "he": "lo", "she": "lo", "it": "lo"}
E_TENSE_SUFFIX = {"present": "", "future": "ru", "past": "se"}
E_PLURAL = "it"
E_LOC_INCORP = {"to": "mo", "into": "nu", "in": "ni"}

# For the polysynthetic mixed language, we need to handle the verb forms with subject prefixes and tense suffixes, noun incorporation for objects and locations, and the irregular copula forms while following the basic SVO order. The verb forms are highly polysynthetic, so we have a subject prefix that indicates person and a tense suffix that indicates tense, with the root verb in between. Noun incorporation allows us to include the object and location nouns directly in the verb complex.
def e_noun(noun: str, plural: bool = False) -> str:
    form = root(noun)
    if plural:
        form += E_PLURAL
    return form

# For location, we incorporate the location noun directly into the verb complex with a special suffix that indicates the location role.
def e_loc(noun: str, role: str) -> str:
    return root(noun) + E_LOC_INCORP[role]

# Finally, we can put everything together to build the polysynthetic mixed verb complex from the Meaning object. We need to include the subject prefix, the root verb, the incorporated object and location nouns, and the tense suffix, all in the correct order.
def e_predicate(
    verb: str,
    subj: str,
    tense: str,
    obj: Optional[str] = None,
    plural_obj: bool = False,
    location: Optional[str] = None,
    location_role: Optional[str] = None,
) -> str:
    parts = [E_SUBJECT_PREFIX[subj]]
    if obj:
        parts.append(e_noun(obj, plural_obj))
    if location:
        parts.append(e_loc(location, location_role))
    parts.append(root(verb))
    if E_TENSE_SUFFIX[tense]:
        parts.append(E_TENSE_SUFFIX[tense])
    return "".join(parts)

# For the polysynthetic mixed language, we follow a similar structure to the other languages, but we need to include the verb forms with subject prefixes and tense suffixes, noun incorporation for objects and locations, and the irregular copula forms while following the basic SVO order. The verb forms are highly polysynthetic, so we have a subject prefix that indicates person and a tense suffix that indicates tense, with the root verb in between. Noun incorporation allows us to include the object and location nouns directly in the verb complex.
def to_polysynthetic_mixed(m: Meaning) -> str:
    if m.clause_type == "weather":
        pred = e_predicate(m.verb, m.subject, m.tense)
        if m.adverb == "tomorrow":
            pred += root("tomorrow")
        elif m.adverb == "yesterday":
            pred += root("yesterday")
        return pred

# The copular clause is also a bit special because it can have either a subject or an object, and the verb "be" is irregular. We also need to handle adjectives and the "very" adverb. The subject prefix is determined by the subject if there is one, or by the object if there is no subject. The adjective comes after the verb, and if there is a "very" adverb, it comes before the adjective.
    if m.clause_type == "copular":
        if m.subject:
            head = E_SUBJECT_PREFIX[m.subject]
        else:
            head = e_noun(m.obj, m.plural_object)
        adj = root(m.adjective)
        if m.adverb == "very":
            adj = root("very") + adj
        subject_for_agreement = m.subject if m.subject else "he"
        return head + adj + E_SUBJECT_PREFIX[subject_for_agreement]

# For causal clauses, we build the main clause and the cause clause separately and then join them with "because". The main clause is built as an intransitive clause with the subject and verb, and the cause clause is built from the Meaning object in m.cause. We join them together with the root for "because".
    if m.clause_type == "causal":
        main = to_polysynthetic_mixed(Meaning("", "intransitive", subject=m.subject, verb=m.verb, tense=m.tense))
        sub = to_polysynthetic_mixed(m.cause)
        return main + root("because") + sub

# For transitive and intransitive clauses, we just need to build the verb complex with the appropriate subject prefix, root verb, incorporated object and location nouns, and tense suffix. The word order is still SVO, but the verb complex can include a lot of information through incorporation and affixation.
    pred = e_predicate(
        verb=m.verb,
        subj=m.subject,
        tense=m.tense,
        obj=m.obj,
        plural_obj=m.plural_object,
        location=m.location,
        location_role=m.location_role,
    )

# Adverbs of place like "here" can just be added at the end of the sentence.
    if m.adverb == "here":
        pred += root("here")
    elif m.adverb == "tomorrow" and m.tense == "future":
        pred += root("tomorrow")
    elif m.adverb == "yesterday" and m.tense == "past":
        pred += root("yesterday")

# Finally, we return the fully constructed verb complex with any adverbs added at the end.
    return pred

## 3. The dataset

In [ ]:
# Now we need to generate a comprehensive dataset of Meaning objects that cover all the different combinations of clause types, verbs, subjects, objects, tenses, adverbs, locations, and pluralities that we want to test our languages on. This dataset should be large enough to thoroughly test the capabilities of each language in expressing the various meanings.
def generate_meanings() -> List[Meaning]:
    data: List[Meaning] = []
    subjects = ["I", "you", "he"]

    # GO
    for subj in subjects:
        for tense, adv in [("present", None), ("future", "tomorrow"), ("past", "yesterday")]:
            data.append(Meaning("", "intransitive", subject=subj, verb="go", tense=tense, adverb=adv))
            data.append(Meaning("", "intransitive", subject=subj, verb="go", tense=tense, adverb=adv,
                                location="city", location_role="to"))
            data.append(Meaning("", "intransitive", subject=subj, verb="go", tense=tense, adverb=adv,
                                location="house", location_role="into"))
        data.append(Meaning("", "intransitive", subject=subj, verb="go", tense="present", adverb="here"))

    # CRY
    for subj in subjects:
        for tense, adv in [("present", None), ("future", "tomorrow"), ("past", "yesterday")]:
            data.append(Meaning("", "intransitive", subject=subj, verb="cry", tense=tense, adverb=adv))
            data.append(Meaning("", "intransitive", subject=subj, verb="cry", tense=tense, adverb=adv,
                                location="city", location_role="in"))
            data.append(Meaning("", "intransitive", subject=subj, verb="cry", tense=tense, adverb=adv,
                                location="house", location_role="in"))
        data.append(Meaning("", "intransitive", subject=subj, verb="cry", tense="present", adverb="here"))

    # RAIN
    for tense, adv in [("present", None), ("future", "tomorrow"), ("past", "yesterday")]:
        data.append(Meaning("", "weather", subject="it", verb="rain", tense=tense, adverb=adv))

    # EAT
    edible_objects = [
        ("food", False),
        ("bread", False),
        ("fruit", False),
        ("meal", False),
        ("meal", True),
    ]
    for subj in subjects:
        for obj, plural in edible_objects:
            for tense, adv in [("present", None), ("future", "tomorrow"), ("past", "yesterday")]:
                data.append(Meaning("", "transitive", subject=subj, verb="eat", obj=obj,
                                    plural_object=plural, tense=tense, adverb=adv))
                data.append(Meaning("", "transitive", subject=subj, verb="eat", obj=obj,
                                    plural_object=plural, tense=tense, adverb=adv,
                                    location="city", location_role="in"))
                data.append(Meaning("", "transitive", subject=subj, verb="eat", obj=obj,
                                    plural_object=plural, tense=tense, adverb=adv,
                                    location="house", location_role="in"))

    # SEE
    see_patterns = [
        ("house", False, None, None),
        ("house", True, None, None),
        ("house", False, "city", "in"),
        ("city", False, None, None),
        ("city", True, None, None),
        ("city", False, "house", "in"),
        ("person", False, None, None),
        ("person", True, None, None),
        ("person", False, "city", "in"),
        ("person", False, "house", "in"),
        ("person", True, "city", "in"),
        ("person", True, "house", "in"),
        ("tree", False, None, None),
        ("tree", True, None, None),
        ("tree", False, "city", "in"),
        ("tree", False, "house", "in"),
        ("tree", True, "city", "in"),
        ("tree", True, "house", "in"),
    ]

    for subj in subjects:
        for obj, plural, loc, role in see_patterns:
            for tense, adv in [("present", None), ("future", "tomorrow"), ("past", "yesterday")]:
                data.append(Meaning("", "transitive", subject=subj, verb="see", obj=obj,
                                    plural_object=plural, tense=tense, adverb=adv,
                                    location=loc, location_role=role))

    # SAD
    for subj in subjects:
        data.append(Meaning("", "copular", subject=subj, adjective="sad", tense="present"))
        data.append(Meaning("", "copular", subject=subj, adjective="sad", tense="present", adverb="very"))

    # BIG / SMALL
    for noun in ["house", "city", "person", "tree"]:
        for adj in ["big", "small"]:
            data.append(Meaning("", "copular", obj=noun, adjective=adj, tense="present"))
            data.append(Meaning("", "copular", obj=noun, adjective=adj, tense="present", plural_object=True))
            data.append(Meaning("", "copular", obj=noun, adjective=adj, tense="present", adverb="very"))

    # CAUSAL
    for subj in subjects:
        cause = Meaning("", "copular", subject=subj, adjective="sad", tense="present")
        data.append(Meaning("", "causal", subject=subj, verb="cry", tense="present", cause=cause))

    # Deduplicate by English gloss
    seen = set()
    out: List[Meaning] = []
    for m in data:
        m.english = build_english(m)
        if m.english not in seen:
            seen.add(m.english)
            out.append(m)

    return out

In [ ]:
# Now that we have all the functions to convert Meaning objects into sentences in each of our artificial languages, we can build a dataset by applying these functions to a comprehensive set of Meaning objects that cover a wide range of linguistic phenomena. We will generate a list of Meaning objects, convert them into sentences in each language, and then compile everything into a structured format for export.
def build_rows(meanings: List[Meaning]) -> List[Dict[str, str]]:
    rows = []
    for idx, m in enumerate(meanings, start=1):
        rows.append({
            "id": str(idx),
            "english": m.english,
            "language_A": to_isolating(m),
            "language_B": to_analytic(m),
            "language_C": to_fusional_prefixing(m),
            "language_D": to_agglutinative_suffixing(m),
            "language_E": to_polysynthetic_mixed(m),
        })
    return rows

# Finally, we can export the dataset in a tab-separated format for easy use in experiments, as well as save it as CSV files and a JSON file for the private key. We will also create a ZIP archive containing all the generated files for easy distribution.
def to_tsv(rows: List[Dict[str, str]]) -> str:
    headers = ["id", "english", "language_A", "language_B", "language_C", "language_D", "language_E"]
    lines = ["\t".join(headers)]
    for r in rows:
        lines.append("\t".join(r[h] for h in headers))
    return "\n".join(lines)

In [ ]:
# The private key contains the typological features and key markers for each language, which can be used for evaluation and interpretation of the results. This key should be kept private to prevent biasing the models during training and evaluation.
language_key = {
    "language_A": {
        "type": "isolating",
        "description": "Free grammatical particles express tense, plurality and location; no prefixes or suffixes.",
        "key_markers": {
            "vek": "future particle",
            "tun": "past particle",
            "nul": "plural particle",
            "du": "to/allative particle",
            "ri": "into/illative particle",
            "na": "in/locative particle",
        },
    },
    "language_B": {
        "type": "analytic",
        "description": "Auxiliary-like words express tense; plurality is expressed through a classifier-like word before nouns.",
        "key_markers": {
            "zo": "future auxiliary",
            "ki": "past auxiliary",
            "enu": "plural classifier",
            "mu": "to/allative adposition",
            "nel": "into/illative adposition",
            "sha": "in/locative adposition",
        },
    },
    "language_C": {
        "type": "fusional prefixing",
        "description": "Single prefixes fuse person and tense; the same tense has different prefixes depending on subject.",
        "key_markers": {
            "me/ti/na": "present prefixes fused with person",
            "uk/ot/ev": "past prefixes fused with person",
            "ar/el/or": "future prefixes fused with person",
            "esh/um/ai/ok/ir": "noun-class-sensitive plural endings",
            "dra": "to/allative marker",
            "vren": "into/illative marker",
            "nim": "in/locative marker",
        },
    },
    "language_D": {
        "type": "agglutinative suffixing",
        "description": "Transparent suffix chains: each suffix corresponds to one stable grammatical function.",
        "key_markers": {
            "ka": "first-person suffix",
            "tu": "second-person suffix",
            "li": "third-person suffix",
            "ra": "future suffix",
            "si": "past suffix",
            "min": "plural suffix",
            "mu": "to/allative case suffix",
            "nun": "into/illative case suffix",
            "ne": "in/locative case suffix",
        },
    },
    "language_E": {
        "type": "polysynthetic mixed",
        "description": "Subject prefixes, incorporated nouns/locations and tense suffixes combine into one complex predicate word.",
        "key_markers": {
            "ma/ta/lo": "subject prefixes",
            "ru": "future suffix inside predicate",
            "se": "past suffix inside predicate",
            "it": "plural suffix on incorporated noun",
            "mo": "to/allative incorporated locative",
            "nu": "into/illative incorporated locative",
            "ni": "in/locative incorporated locative",
        },
    },
}

In [ ]:
# The main function to generate the dataset, build the rows, and export everything to files.
if __name__ == "__main__":
    data = generate_meanings()
    rows = build_rows(data)
    df = pd.DataFrame(rows)

    print(to_tsv(rows))
    print(f"\nGenerated {len(rows)} sentences.")

    out_dir = "artificial_language_experiment_typological"
    os.makedirs(out_dir, exist_ok=True)

    # Full dataset with English glosses
    df.to_csv(os.path.join(out_dir, "dataset_with_english.csv"), index=False)

    # Dataset without English glosses
    df_without_english = df[["id", "language_A", "language_B", "language_C", "language_D", "language_E"]]
    df_without_english.to_csv(os.path.join(out_dir, "dataset_without_english.csv"), index=False)

    # Private key for evaluation and interpretation
    with open(os.path.join(out_dir, "language_key_private.json"), "w", encoding="utf-8") as f:
        json.dump(language_key, f, ensure_ascii=False, indent=2)

    # Compact typological summary
    typology_summary = pd.DataFrame([
        {"language": lang, "type": info["type"], "description": info["description"]}
        for lang, info in language_key.items()
    ])
    typology_summary.to_csv(os.path.join(out_dir, "typology_summary_private.csv"), index=False)

    # Zip everything
    shutil.make_archive("artificial_language_experiment_typological", "zip", out_dir)

    print(f"Files saved in: {out_dir}")
    print("\nFiles created:")
    for f in sorted(os.listdir(out_dir)):
        print("-", f)
    print("\nZIP archive created: artificial_language_experiment_typological.zip")
